<a href="https://colab.research.google.com/github/Meghana-200502/NNDL/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# ==========================================
# 1) Install dependencies
# ==========================================
!pip -q install pandas pyarrow numpy scikit-learn matplotlib torch

# ==========================================
# 2) Imports
# ==========================================
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 3) Reproducibility
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================================
# 4) File paths
# ==========================================
CITIES_PATH = "/content/cities.csv"
COUNTRIES_PATH = "/content/countries.csv"
WEATHER_PATH = "/content/daily_weather.parquet"

assert os.path.exists(CITIES_PATH), f"Missing file: {CITIES_PATH}"
assert os.path.exists(COUNTRIES_PATH), f"Missing file: {COUNTRIES_PATH}"
assert os.path.exists(WEATHER_PATH), f"Missing file: {WEATHER_PATH}"

# ==========================================
# 5) Load files
# ==========================================
cities_df = pd.read_csv(CITIES_PATH)
countries_df = pd.read_csv(COUNTRIES_PATH)
weather_df = pd.read_parquet(WEATHER_PATH)

print("cities.csv shape      :", cities_df.shape)
print("countries.csv shape   :", countries_df.shape)
print("daily_weather shape   :", weather_df.shape)

print("\nCities columns:")
print(cities_df.columns.tolist())

print("\nCountries columns:")
print(countries_df.columns.tolist())

print("\nWeather columns:")
print(weather_df.columns.tolist())

display(weather_df.head())

# ==========================================
# 6) Helper functions to detect columns
# ==========================================
def find_first_matching_column(columns, candidates):
    cols_lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in columns:
        c_low = c.lower()
        for cand in candidates:
            if cand.lower() in c_low:
                return c
    return None

def pick_date_col(df):
    return find_first_matching_column(
        df.columns,
        ["date", "datetime", "time", "timestamp", "day"]
    )

def pick_city_col(df):
    return find_first_matching_column(
        df.columns,
        ["city", "city_name", "name", "location", "station", "station_name"]
    )

def pick_country_col(df):
    return find_first_matching_column(
        df.columns,
        ["country", "country_name", "country_id", "iso2", "iso3"]
    )

def pick_target_temp_col(df):
    return find_first_matching_column(
        df.columns,
        [
            "temperature", "temp", "avg_temp", "mean_temp", "meantemp",
            "tavg", "daily_avg_temp", "temperature_2m_mean"
        ]
    )

def numeric_columns(df):
    return df.select_dtypes(include=[np.number]).columns.tolist()

# ==========================================
# 7) Detect key columns
# ==========================================
date_col = pick_date_col(weather_df)
city_col = pick_city_col(weather_df)
country_col = pick_country_col(weather_df)
target_col = pick_target_temp_col(weather_df)

print("\nDetected columns:")
print("date_col   =", date_col)
print("city_col   =", city_col)
print("country_col=", country_col)
print("target_col =", target_col)

if date_col is None:
    raise ValueError("Could not detect a date column in daily_weather.parquet")

if target_col is None:
    raise ValueError("Could not detect a temperature column in daily_weather.parquet")

# ==========================================
# 8) Convert date and sort
# ==========================================
weather_df[date_col] = pd.to_datetime(weather_df[date_col], errors="coerce")
weather_df = weather_df.dropna(subset=[date_col]).copy()
weather_df = weather_df.sort_values(date_col).reset_index(drop=True)

# ==========================================
# 9) Optional: choose one city if multiple exist
# ==========================================
if city_col is not None and weather_df[city_col].nunique() > 1:
    print("\nMultiple cities found. Top cities by row count:")
    city_counts = weather_df[city_col].value_counts()
    display(city_counts.head(10))

    # Automatically use the city with the most rows
    selected_city = city_counts.index[0]
    print(f"Using city: {selected_city}")
    weather_df = weather_df[weather_df[city_col] == selected_city].copy()
    weather_df = weather_df.sort_values(date_col).reset_index(drop=True)
else:
    selected_city = None

# ==========================================
# 10) Keep useful columns
# ==========================================
# Use numeric columns as features, excluding obvious IDs where possible.
num_cols = numeric_columns(weather_df)

# Remove columns that are clearly identifiers
id_like_cols = []
for c in num_cols:
    c_low = c.lower()
    if any(x in c_low for x in ["id", "index", "zip", "postal", "lat", "lon", "latitude", "longitude"]):
        id_like_cols.append(c)

feature_cols = [c for c in num_cols if c not in id_like_cols]

# Ensure target is included
if target_col not in feature_cols:
    feature_cols.append(target_col)

# Try to keep only a compact set of weather-related features if present
preferred_feature_keywords = [
    "temp", "humid", "pressure", "wind", "rain", "precip",
    "cloud", "dew", "visibility", "solar"
]

preferred_cols = []
for c in feature_cols:
    c_low = c.lower()
    if any(k in c_low for k in preferred_feature_keywords):
        preferred_cols.append(c)

# If enough weather-like columns are found, use them; otherwise use numeric columns
if len(preferred_cols) >= 2:
    feature_cols = sorted(list(set(preferred_cols + [target_col])))

print("\nFinal feature columns:")
print(feature_cols)
print("Target column:", target_col)

# ==========================================
# 11) Clean missing values
# ==========================================
work_df = weather_df[[date_col] + feature_cols].copy()
work_df = work_df.replace([np.inf, -np.inf], np.nan)

# First, attempt forward and backward fill for time-series context
work_df = work_df.fillna(method="ffill").fillna(method="bfill")

# If any NaNs still remain (e.g., column was entirely NaN), fill with mean for numeric columns
for col in feature_cols:
    if work_df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(work_df[col]):
            col_mean = work_df[col].mean()
            if pd.isna(col_mean): # If the column was entirely NaN, its mean will be NaN
                work_df[col] = work_df[col].fillna(0) # Fill with 0 as a fallback
            else:
                work_df[col] = work_df[col].fillna(col_mean)

# Drop any remaining rows with NaNs (should only be non-numeric or unfillable date issues now)
work_df = work_df.dropna().reset_index(drop=True)

print("\nPrepared data shape:", work_df.shape)
display(work_df.head())

# ==========================================
# 12) Train/test split by time
# ==========================================
# Time-series split: first 80% train, last 20% test
split_idx = int(len(work_df) * 0.8)

train_df = work_df.iloc[:split_idx].copy()
test_df = work_df.iloc[split_idx:].copy()

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))

# ==========================================
# 13) Scale features and target
# ==========================================
x_scaler = StandardScaler()
y_scaler = StandardScaler()

train_X = train_df[feature_cols].values
test_X  = test_df[feature_cols].values

train_y = train_df[[target_col]].values
test_y  = test_df[[target_col]].values

train_X_scaled = x_scaler.fit_transform(train_X)
test_X_scaled  = x_scaler.transform(test_X)

train_y_scaled = y_scaler.fit_transform(train_y)
test_y_scaled  = y_scaler.transform(test_y)

# ==========================================
# 14) Create sequences for next-value prediction
# ==========================================
SEQ_LEN = 30

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])  # next day target
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_train, y_train = make_sequences(train_X_scaled, train_y_scaled, SEQ_LEN)

# For test sequences, prepend the last training context
combined_X = np.vstack([train_X_scaled[-SEQ_LEN:], test_X_scaled])
combined_y = np.vstack([train_y_scaled[-SEQ_LEN:], test_y_scaled])
X_test, y_test = make_sequences(combined_X, combined_y, SEQ_LEN)

print("\nSequence shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)

if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError("Not enough rows to build sequences. Reduce SEQ_LEN or check data size.")

# ==========================================
# 15) Dataset / DataLoader
# ==========================================
class WeatherDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32

train_loader = DataLoader(WeatherDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(WeatherDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 16) Define vanilla RNN model
# ==========================================
class NextValueRNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            nonlinearity="tanh",
            batch_first=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x, return_hidden_sequence=False):
        # x shape: (batch, seq_len, input_size)
        out, h_n = self.rnn(x)           # out: (batch, seq_len, hidden_size)
        last_hidden = out[:, -1, :]      # final time step
        pred = self.fc(last_hidden)      # next value

        if return_hidden_sequence:
            return pred, out
        return pred

model = NextValueRNN(input_size=len(feature_cols), hidden_size=64, num_layers=1).to(device)
print(model)

# ==========================================
# 17) Training helpers
# ==========================================
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 50

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    preds_all, targets_all = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)

            total_loss += loss.item() * xb.size(0)
            preds_all.append(preds.cpu().numpy())
            targets_all.append(yb.cpu().numpy())

    preds_all = np.vstack(preds_all)
    targets_all = np.vstack(targets_all)
    return total_loss / len(loader.dataset), preds_all, targets_all

# ==========================================
# 18) Train
# ==========================================
train_losses = []
test_losses = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    test_loss, _, _ = evaluate(model, test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")

# ==========================================
# 19) Plot learning curves
# ==========================================
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")
plt.title("RNN Training Curve")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.show()

# ==========================================
# 20) Final predictions in original temperature units
# ==========================================
test_loss, pred_scaled, true_scaled = evaluate(model, test_loader)

pred = y_scaler.inverse_transform(pred_scaled)
true = y_scaler.inverse_transform(true_scaled)

mae = mean_absolute_error(true, pred)
rmse = np.sqrt(mean_squared_error(true, pred))

print(f"Test MAE  : {mae:.4f}")
print(f"Test RMSE : {rmse:.4f}")

plt.figure(figsize=(12, 5))
plt.plot(true, label="Actual")
plt.plot(pred, label="Predicted")
plt.title(f"Next-Day Temperature Prediction ({target_col})")
plt.xlabel("Test Time Index")
plt.ylabel("Temperature")
plt.legend()
plt.grid(True)
plt.show()

results_df = pd.DataFrame({
    "actual": true.flatten(),
    "predicted": pred.flatten()
})
display(results_df.head(20))

# ==========================================
# 21) Memory analysis: input-gradient saliency
# Shows which past timesteps matter most to the prediction
# ==========================================
model.eval()

sample_idx = min(10, len(X_test) - 1)
x_sample = torch.tensor(
    X_test[sample_idx:sample_idx+1],
    dtype=torch.float32,
    device=device,
    requires_grad=True
)

pred_sample = model(x_sample)
pred_sample.backward()

grads = x_sample.grad.detach().cpu().numpy()[0]         # (seq_len, features)
timestep_importance = np.mean(np.abs(grads), axis=1)    # average over features

plt.figure(figsize=(10, 4))
plt.plot(range(1, SEQ_LEN + 1), timestep_importance, marker='o')
plt.title("Past-Timestep Importance for Prediction")
plt.xlabel("Timestep in Input Window (1 = oldest, 30 = most recent)")
plt.ylabel("Average absolute gradient")
plt.grid(True)
plt.show()

# ==========================================
# 22) Hidden-state analysis
# See how hidden state evolves through the sequence
# ==========================================
with torch.no_grad():
    _, hidden_seq = model(
        torch.tensor(X_test[sample_idx:sample_idx+1], dtype=torch.float32, device=device),
        return_hidden_sequence=True
    )

hidden_seq = hidden_seq.squeeze(0).cpu().numpy()  # (seq_len, hidden_size)
hidden_norm = np.linalg.norm(hidden_seq, axis=1)

plt.figure(figsize=(10, 4))
plt.plot(range(1, SEQ_LEN + 1), hidden_norm, marker='o')
plt.title("Hidden-State Magnitude Across Timesteps")
plt.xlabel("Timestep")
plt.ylabel("||hidden_state||")
plt.grid(True)
plt.show()

# ==========================================
# 23) Sequence length experiment
# Tests how memory behaves as history window grows
# ==========================================
def run_seq_experiment(seq_len, hidden_size=64, epochs=20):
    X_train_local, y_train_local = make_sequences(train_X_scaled, train_y_scaled, seq_len)

    combined_X_local = np.vstack([train_X_scaled[-seq_len:], test_X_scaled])
    combined_y_local = np.vstack([train_y_scaled[-seq_len:], test_y_scaled])
    X_test_local, y_test_local = make_sequences(combined_X_local, combined_y_local, seq_len)

    train_loader_local = DataLoader(
        WeatherDataset(X_train_local, y_train_local),
        batch_size=32,
        shuffle=True
    )
    test_loader_local = DataLoader(
        WeatherDataset(X_test_local, y_test_local),
        batch_size=32,
        shuffle=False
    )

    model_local = NextValueRNN(input_size=len(feature_cols), hidden_size=hidden_size).to(device)
    optimizer_local = torch.optim.Adam(model_local.parameters(), lr=1e-3)
    criterion_local = nn.MSELoss()

    for _ in range(epochs):
        model_local.train()
        for xb, yb in train_loader_local:
            xb, yb = xb.to(device), yb.to(device)
            optimizer_local.zero_grad()
            preds = model_local(xb)
            loss = criterion_local(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_local.parameters(), 1.0)
            optimizer_local.step()

    model_local.eval()
    preds_all, targets_all = [], []
    with torch.no_grad():
        for xb, yb in test_loader_local:
            xb, yb = xb.to(device), yb.to(device)
            preds = model_local(xb)
            preds_all.append(preds.cpu().numpy())
            targets_all.append(yb.cpu().numpy())

    preds_all = np.vstack(preds_all)
    targets_all = np.vstack(targets_all)

    preds_inv = y_scaler.inverse_transform(preds_all)
    targets_inv = y_scaler.inverse_transform(targets_all)

    mae_local = mean_absolute_error(targets_inv, preds_inv)
    rmse_local = np.sqrt(mean_squared_error(targets_inv, preds_inv))
    return mae_local, rmse_local

seq_lengths = [7, 14, 30, 60]
maes, rmses = [], []

for s in seq_lengths:
    if len(train_df) > s + 5 and len(test_df) > 5:
        mae_s, rmse_s = run_seq_experiment(seq_len=s, hidden_size=64, epochs=20)
        maes.append(mae_s)
        rmses.append(rmse_s)
        print(f"SEQ_LEN={s:2d} -> MAE={mae_s:.4f}, RMSE={rmse_s:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(seq_lengths[:len(rmses)], rmses, marker='o')
plt.title("Effect of Sequence Length on RNN Performance")
plt.xlabel("Sequence Length")
plt.ylabel("RMSE")
plt.grid(True)
plt.show()

# ==========================================
# 24) Simple interpretation text
# ==========================================
print("\nInterpretation:")
print("- If recent timesteps have larger gradient importance, the RNN relies more on recent weather.")
print("- If importance quickly drops for older timesteps, the vanilla RNN is forgetting older information.")
print("- If longer sequence length does not improve RMSE, long-range memory is weak.")

Using device: cpu
cities.csv shape      : (1245, 8)
countries.csv shape   : (214, 11)
daily_weather shape   : (27635763, 14)

Cities columns:
['station_id', 'city_name', 'country', 'state', 'iso2', 'iso3', 'latitude', 'longitude']

Countries columns:
['country', 'native_name', 'iso2', 'iso3', 'population', 'area', 'capital', 'capital_lat', 'capital_lng', 'region', 'continent']

Weather columns:
['station_id', 'city_name', 'date', 'season', 'avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'snow_depth_mm', 'avg_wind_dir_deg', 'avg_wind_speed_kmh', 'peak_wind_gust_kmh', 'avg_sea_level_pres_hpa', 'sunshine_total_min']


,station_id,city_name,date,season,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,snow_depth_mm,avg_wind_dir_deg,avg_wind_speed_kmh,peak_wind_gust_kmh,avg_sea_level_pres_hpa,sunshine_total_min
0,41515,Asadabad,1957-07-01,Summer,27.0,21.1,35.6,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,41515,Asadabad,1957-07-02,Summer,22.8,18.9,32.2,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,41515,Asadabad,1957-07-03,Summer,24.3,16.7,35.6,1.0,NaN,NaN,NaN,NaN,NaN,NaN
3,41515,Asadabad,1957-07-04,Summer,26.6,16.1,37.8,4.1,NaN,NaN,NaN,NaN,NaN,NaN
4,41515,Asadabad,1957-07-05,Summer,30.8,20.0,41.7,0.0,NaN,NaN,NaN,NaN,NaN,NaN



Detected columns:
date_col   = date
city_col   = city_name
country_col= None
target_col = avg_temp_c

Multiple cities found. Top cities by row count:


,count
city_name,
Brussels,69347
Victoria,64679
Haarlem,63070
Vienna,61477
Visby,59935
Stockholm,59774
Linköping,59668
Falun,59634
Zagreb,59084


Using city: Brussels

Final feature columns:
['avg_temp_c', 'avg_wind_dir_deg', 'avg_wind_speed_kmh', 'max_temp_c', 'min_temp_c', 'peak_wind_gust_kmh', 'precipitation_mm']
Target column: avg_temp_c

Prepared data shape: (69347, 8)


,date,avg_temp_c,avg_wind_dir_deg,avg_wind_speed_kmh,max_temp_c,min_temp_c,peak_wind_gust_kmh,precipitation_mm
0,1833-01-02,1.5,234.0,15.9,-1.4,-4.8,0.0,0.4
1,1833-01-03,1.5,234.0,15.9,-3.1,-6.8,0.0,0.4
2,1833-01-04,1.5,234.0,15.9,-3.9,-6.8,0.0,0.4
3,1833-01-05,1.5,234.0,15.9,-4.4,-9.0,0.0,0.4
4,1833-01-06,1.5,234.0,15.9,1.4,-8.5,0.0,0.4


Train rows: 55477
Test rows : 13870

Sequence shapes:
X_train: (55447, 30, 7) y_train: (55447, 1)
X_test : (13870, 30, 7) y_test : (13870, 1)
NextValueRNN(
  (rnn): RNN(7, 64, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)
Epoch 001 | Train Loss: 0.063301 | Test Loss: 7.369878
Epoch 010 | Train Loss: 0.036048 | Test Loss: 7.030747


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6062/3800515350.py", line 380, in <cell line: 0>
    train_loss = train_one_epoch(model, train_loader)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6062/3800515350.py", line 344, in train_one_epoch
    preds = model(xb)
            ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6062/3800515350.py", line 318, in forward
    out, h_n = self.rnn(x)           # out: (batch, seq_len, 

TypeError: object of type 'NoneType' has no len()